# desCONOCIDOS

In [4]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from collections import Counter, deque

class FaceEmbeddingModel(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()

        backbone = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        num_features = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.embedding = nn.Linear(num_features, embedding_dim)
        self.bn = nn.BatchNorm1d(embedding_dim)

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.bn(self.embedding(features))

        # Normalización L2 (en float32 por estabilidad con AMP)
        return F.normalize(embedding.float(), p=2, dim=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FaceEmbeddingModel(128)

checkpoint = torch.load("best_face_embedding_resnet18_arcface.pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\aserg/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 59.0MB/s]


In [6]:
# Carga del modelo y galería (movemos los embeddings a GPU UNA sola vez)
dnn_model = "deploy.prototxt.txt"
dnn_weights = "res10_300x300_ssd_iter_140000.caffemodel"
net = cv2.dnn.readNetFromCaffe(dnn_model, dnn_weights)

gallery = torch.load("gallery_embeddings_arcface.pt", map_location="cpu")
gallery_embeddings = gallery["embeddings"].to(device)
filenames = gallery["filenames"]

model.eval()

# Ventana deslizante: solo recuerda los últimos 20 frames
votation = deque(maxlen=20)

def get_celeba_crop(frame, box):
    H, W = frame.shape[:2]
    x1, y1, x2, y2 = box

    w = x2 - x1
    h = y2 - y1
    cx = x1 + w / 2.0
    cy = y1 + h / 2.0

    new_h = h * 1.65
    new_w = new_h * (178.0 / 218.0)

    cy_shifted = cy - (h * 0.08)

    nx1 = int(max(0, cx - new_w / 2.0))
    ny1 = int(max(0, cy_shifted - new_h / 2.0))
    nx2 = int(min(W, cx + new_w / 2.0))
    ny2 = int(min(H, cy_shifted + new_h / 2.0))

    return frame[ny1:ny2, nx1:nx2], (nx1, ny1, nx2, ny2)

def get_most_similar_famous(face):
    face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
    face_image = Image.fromarray(face_rgb)

    face_tensor = transform(face_image).unsqueeze(0).to(device)

    with torch.no_grad():
        query_embedding = model(face_tensor)
        query_embedding = F.normalize(query_embedding, p=2, dim=1)

    similarities = (query_embedding @ gallery_embeddings.T).squeeze(0)

    best_index = torch.argmax(similarities).item()

    route = "." + filenames[best_index]
    votation.append(route)

    most_common_route, _ = Counter(votation).most_common(1)[0]
    return most_common_route

video_capture = cv2.VideoCapture(0)

while True:
    ret, frame = video_capture.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    H, W = frame.shape[:2]

    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), [104.0, 117.0, 123.0], False, False)
    net.setInput(blob)
    detections = net.forward()

    best_idx = np.argmax(detections[0, 0, :, 2])
    max_confidence = detections[0, 0, best_idx, 2]

    if max_confidence > 0.75:
        box = detections[0, 0, best_idx, 3:7] * np.array([W, H, W, H])
        face_crop, (nx1, ny1, nx2, ny2) = get_celeba_crop(frame, box)

        cv2.rectangle(frame, (nx1, ny1), (nx2, ny2), (0, 255, 0), 2)

        if face_crop.size > 0:
            found = get_most_similar_famous(face_crop)
        if found is not None:
            overlay = cv2.imread(found)
            if overlay is not None:
                h, w = overlay.shape[:2]
                x = max(0, min((W - w) // 2, W - w))
                y = max(0, min(20, H - h))
                frame[y : y + h, x : x + w] = overlay
    else:
        votation.clear()

    cv2.putText(frame, "Presiona 'ESC' para salir",
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
    cv2.imshow('Face Detection', frame)

    if (cv2.waitKey(1) & 0xFF) == 27:
        break

video_capture.release()
cv2.destroyAllWindows()

# Nose dibujarlo

In [ ]:
import time
import math
import cv2
import mediapipe as mp
import numpy as np

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

latest = None
latest_hands = None

# Índices de los landmarks para cada ojo
LEFT_EYE = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]

# Parámetros de detección
EAR_THRESHOLD = 0.26      # Umbral por debajo del cual el ojo se considera cerrado
BLINK_FRAMES = 18         # Fotogramas seguidos necesarios para considerar el parpadeo suficientemente largo
GRAB_THRESHOLD = 0.25     # Distancia pulgar-índice para "agarrar"
SELECT_RADIUS = 50        # Radio en píxeles para seleccionar un trazo
ATTACH_DISTANCE = 2.0     # Distancia máxima al centro de la cara (en unidades de "ancho entre ojos")


def on_result(result, output_image, timestamp_ms):
    global latest
    latest = result

def hand_on_result(result, output_image, timestamp_ms):
    global latest_hands
    latest_hands = result

def calculate_ear(landmarks, indexes, w, h):
    # Convertir coordenadas normalizadas a píxeles
    pts = np.array([(landmarks[i].x * w, landmarks[i].y * h) for i in indexes])

    # Distancias verticales
    v1 = np.linalg.norm(pts[1] - pts[5])
    v2 = np.linalg.norm(pts[2] - pts[4])

    # Distancia horizontal
    horiz = np.linalg.norm(pts[0] - pts[3])

    return (v1 + v2) / (2.0 * horiz)

def nearest_line(x, y, radius=SELECT_RADIUS):
    for i, line in enumerate(lines):
        # Distancia euclidiana de (x, y) a todos los puntos del trazo
        distancias = np.linalg.norm(line - np.array([x, y]), axis=1)
        if np.min(distancias) < radius:
            return i
    return None

def calculate_distance(p1, p2):
    return math.sqrt((p1.x - p2.x) ** 2 + (p1.y - p2.y) ** 2)

def is_user_grabbing(landmarks, w, h, umbral=GRAB_THRESHOLD):
    thumb = landmarks[4]
    forefinger = landmarks[8]
    wrist = landmarks[0]
    media_base = landmarks[9]

    tip_distance = calculate_distance(thumb, forefinger)
    palm_size = calculate_distance(wrist, media_base)

    if palm_size == 0:
        return None, None, False

    pixel_x = int((thumb.x + forefinger.x) / 2 * w)
    pixel_y = int((thumb.y + forefinger.y) / 2 * h)
    distance = tip_distance / palm_size

    return pixel_x, pixel_y, distance < umbral

def face_frame(face, w, h):
    a, b = face[33], face[263]
    pa = np.array([a.x * w, a.y * h])
    pb = np.array([b.x * w, b.y * h])
    v = pb - pa
    scale = np.linalg.norm(v)
    if scale < 1e-6:
        return None
    return (pa + pb) / 2.0, scale, math.atan2(v[1], v[0])

def rotation(angle):
    c, s = math.cos(angle), math.sin(angle)
    return np.array([[c, -s], [s, c]])

def to_local(points, frame):
    origin, scale, angle = frame
    return ((points - origin) @ rotation(angle)) / scale

def to_world(local, frame):
    origin, scale, angle = frame
    return origin + scale * (local @ rotation(angle).T)

def release_line(i, frame):
    if i is None or i >= len(lines) or frame is None:
        return
    origin, scale, _ = frame
    d = np.min(np.linalg.norm(lines[i] - origin, axis=1))
    if d < ATTACH_DISTANCE * scale:
        anchors[i] = to_local(lines[i].astype(np.float64), frame)

face_options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_faces=1,
    output_face_blendshapes=True,
    result_callback=on_result,
)

hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_hands=2,
    result_callback=hand_on_result,
)

cap = cv2.VideoCapture(0)

drawing = False
frame_counter = 0
last_ts = 0

lines = []
anchors = []
current_line = []
selected_line = None
prev_pos = None

with (
    FaceLandmarker.create_from_options(face_options) as face_landmarker,
    HandLandmarker.create_from_options(hand_options) as hand_landmarker,
):
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        frame_ref = None

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        ts = max(int(time.time() * 1000), last_ts + 1)
        last_ts = ts
        face_landmarker.detect_async(mp_image, ts)
        hand_landmarker.detect_async(mp_image, ts)

        if latest is not None and latest.face_landmarks:
            face = latest.face_landmarks[0]
            frame_ref = face_frame(face, w, h)

            # Actualiza los trazos anclados para que sigan a la cara
            if frame_ref is not None:
                for i, local in enumerate(anchors):
                    if local is not None and i != selected_line:
                        lines[i] = to_world(local, frame_ref).astype(np.int32)

            left_ear = calculate_ear(face, LEFT_EYE, w, h)
            right_ear = calculate_ear(face, RIGHT_EYE, w, h)
            ear_prom = (left_ear + right_ear) / 2.0

            # Activación de dibujo por parpadeo
            if ear_prom < EAR_THRESHOLD:
                frame_counter += 1
            else:
                if frame_counter >= BLINK_FRAMES:
                    drawing = not drawing
                    if len(current_line) > 1:
                        lines.append(np.array(current_line, dtype=np.int32))
                        anchors.append(None)
                    current_line = []
                    selected_line = None
                    prev_pos = None
                frame_counter = 0

            if drawing:
                nose = face[4]
                x = int(nose.x * w)
                y = int(nose.y * h)

                # Landmark de la nariz como referencia
                cv2.circle(frame, (x, y), 6, (0, 255, 0), -1)
                current_line.append([x, y])

        if not drawing and lines and latest_hands is not None and latest_hands.hand_landmarks:
            hand = latest_hands.hand_landmarks[0]
            x, y, grabbing = is_user_grabbing(hand, w, h)

            if grabbing and x is not None:
                cv2.circle(frame, (x, y), 8, (0, 0, 255), -1)

                if selected_line is None:
                    selected_line = nearest_line(x, y)
                    if selected_line is not None:
                        anchors[selected_line] = None
                        prev_pos = (x, y)
                    else:
                        prev_pos = None
                elif selected_line < len(lines):
                    # Mueve el trazo según el desplazamiento de la mano
                    if prev_pos is not None:
                        delta = np.array([x - prev_pos[0], y - prev_pos[1]], dtype=np.int32)
                        lines[selected_line] += delta
                    prev_pos = (x, y)
            else:
                release_line(selected_line, frame_ref)
                selected_line = None
                prev_pos = None
        elif not drawing:
            release_line(selected_line, frame_ref)
            selected_line = None
            prev_pos = None

        for i, trazo in enumerate(lines):
            if i == selected_line:
                color = (0, 165, 255)
            elif anchors[i] is not None:
                color = (255, 200, 0)
            else:
                color = (0, 255, 0)
            cv2.polylines(frame, [trazo], isClosed=False, color=color, thickness=5)

        # Trazo en curso
        if len(current_line) > 1:
            cv2.polylines(frame, [np.array(current_line, dtype=np.int32)],
                          isClosed=False, color=(0, 255, 0), thickness=5)

        cv2.imshow("Nose dibujarlo", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('c'):          # Borrar todo
            lines = []
            anchors = []
            drawing = False
            selected_line = None
            prev_pos = None
            current_line = []
        elif key == ord('x'):        # Cancelar el trazo en curso
            drawing = False
            current_line = []
        elif key == ord('u'):        # Deshacer el último trazo
            if lines:
                lines.pop()
                anchors.pop()
            selected_line = None
            prev_pos = None
            drawing = False
            current_line = []
        elif key == 27:              # Esc
            break

cap.release()
cv2.destroyAllWindows()